# ANÁLISIS COMPLETO "clientes_credito"

## Entendimiento del negocio

**ID:** ID de cada cliente\
**LIMIT_BAL:** Monto del crédito otorgado en dólares NT (incluye crédito individual y familiar/complementario)\
**SEXO:** Género (1=masculino, 2=femenino)\
**EDUCACIÓN:** (1=posgrado, 2=universitario, 3=preparatoria, 4=otros)\
**MATRIMONIO:** Estado civil (1=casado, 2=soltero, 3=otros)\
**EDAD:** Edad en años\
**PAY_0:**\
-2 = No consumo ese mes\
-1 = Pagó a tiempo\
0 = Uso crédito y pagó a tiempo\
1 = 1 mes de retraso\
2 = 2 meses de retraso ...\
**BILL_AMT1:** Monto de factura de cada mes (cuánto debía)\
**PAY_AMT1:** Cuánto pagó ese mes

## Librerías

In [ ]:
''' Importar Librerías '''

# Dataset
import numpy as np
import pyodbc
import pandas as pd # Manipulación de datos

# Visualización de datos
import matplotlib.pyplot as plt 
import seaborn as sns

# Partición de datos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler # Escalado de características
from sklearn.decomposition import PCA   # Reducción de dimensionalidad con PCA
import umap # Visualización y reducción de dimensionalidad usando umap

# APRENDIZAJE SUPERVISADO
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, mean_squared_error, roc_curve, auc)

# APRENDIZAJE NO SUPERVISADO
# K-Means, DBSCAN, Hierarchical Clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
# Evaluación de la calidad del clustering
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score)


## Extracción desde SQL

In [ ]:
print(pyodbc.drivers())

**⚠️ Mejora importante (mentalidad pro)**

Ese código funciona, pero en trabajo real deberías:

* No usar SELECT * (mejor seleccionar columnas)
* Manejar errores (try/except)
* Cerrar conexión (conn.close())

In [ ]:
# Extracción de datos desde SQL Server 
with pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=riesgo_credito_db;"
    "Trusted_Connection=yes"
    #UID=usuario; #Cuando no es trusted
    #PWD=contraseña;
) as conn: 
    query= "SELECT * FROM clientes_credito"
    df= pd.read_sql(query, conn)
    
df.head()

In [ ]:
print(df.info())
print(df.describe().round(2))
print(df.nunique())

## 3. Limpieza

In [ ]:
df.drop_duplicates(inplace=True)
df.drop('ID', axis=1, inplace=True)
print(df.isnull().sum())

### Analisis de Variables Categóricas

|EDUCATION:          |   MARRIAGE:   |
|--------------------|---------------|
|1 = graduate school |   1 = married |
|2 = university      |   2 = single  |
|3 = high school     |   3 = others  |
|4 = others          |               |

In [ ]:

# Evaluar la distribución de las variables categóricas
print("EDUCATION:\n", df['EDUCATION'].value_counts())
print("\nMARRIAGE:\n", df['MARRIAGE'].value_counts())
print("\nSEX:\n", df['SEX'].value_counts())
print("\nPAY_0:\n", df['PAY_0'].value_counts())

In [ ]:
# Agrupar 0, 5 y 6 dentro de “others”
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

# Agrupar MARRIAGE 0 dentro de “others”
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

# Mostrar las distribuciones actualizadas
print("EDUCATION:\n", df['EDUCATION'].value_counts())
print("\nMARRIAGE:\n", df['MARRIAGE'].value_counts())

## Análisis Exploratorio (EDA)

### Analisis de Variables Numéricas

In [ ]:
# Agrupar por sexo y calcular estadísticas descriptivas para LIMIT_BAL
estadisticas = df.groupby('SEX')['LIMIT_BAL'].agg(
    ['mean', 'std', 'min', 'max']).round(2)

# Montos más altos de crédito
monto_alto = df.sort_values('LIMIT_BAL', ascending=False).head(10)

# agrupar por edad y ver el promedio y el mayor monto del crédito
promedio_edad = df.groupby('AGE')['LIMIT_BAL'].agg(
    ['mean', 'min' ,'max']).round(2).sort_values(
        'mean', ascending=False).head(10)

print("Estadísticas por sexo:", estadisticas)
print("\nMontos más altos de crédito:", monto_alto)
print("\nPromedio de crédito por edad:", promedio_edad)

### Analisis de Variables Categóricas

In [ ]:
# Porcentaje de la DISTRIBUCIÓN de la variable objetivo
print(df['default_payment_next_month'].value_counts(normalize=True) * 100)

# Visualización de la variable objetivo
sns.countplot(x='default_payment_next_month', data=df)
plt.title('Distribución de Default Payment Next Month')
plt.grid(True)
plt.show()

# Relación entre riesgo y comportamiento de pago
# PAY_0  vs  default_payment_next_month
# clientes con retrasos altos deberían tener mayor probabilidad de default
plt.figure(figsize=(8, 6))
sns.countplot(x='PAY_0', hue='default_payment_next_month', data=df)
plt.title('Relación entre Retraso en Pago y Default')
plt.xlabel('Retraso en Pago (PAY_0)')
plt.ylabel('Count')
plt.grid(True)
plt.show()

# Correlación entre variables de retraso PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6
# Muestran persistencia de morosidad.
# Si alguien se atrasa este mes, probablemente también el siguiente.

# Calcula una matriz de correlación solo de esas variables y grafícala con heatmap
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
plt.figure(figsize=(10, 8))
sns.heatmap(df[pay_cols].corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlación entre Variables de Retraso')
plt.show()

'''Si aparecen inf / NaN, simplemente los filtras o los conviertes en 0.'''

# Limpiar datos para evitar división por cero
df['BILL_AMT1'] = df['BILL_AMT1'].replace(0, 1)  # Reemplaza 0 por 1 para evitar división por cero

# Qué proporción de la deuda paga el cliente.
df['ratio_pago'] = df['PAY_AMT1'] / df['BILL_AMT1']
print(df['ratio_pago'].describe().round(2))

# Visualización de la distribución de variables categóricas
'''SEX, EDUCATION, MARRIAGE, PAY_0'''
plt.figure(figsize=(12, 8))
plt.suptitle('Distribución de Variables Categóricas', fontsize=16)
plt.subplot(2, 2, 1); sns.countplot(x='SEX', data=df)
plt.subplot(2, 2, 2); sns.countplot(x='EDUCATION', data=df)
plt.subplot(2, 2, 3); sns.countplot(x='MARRIAGE', data=df)
plt.subplot(2, 2, 4); sns.countplot(x='PAY_0', data=df)
plt.tight_layout()
plt.grid(True)
plt.show()

### Histogramas, Boxplots, Correlaciones, Pairplots y Distribuciones

1️⃣ Distribución de variables → histogramas\
2️⃣ Detección de outliers → boxplot\
3️⃣ Relaciones globales → matriz de correlación\
4️⃣ Relaciones específicas → pairplot

In [ ]:
num_cols = ['LIMIT_BAL', 'AGE', 
            'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'] 


In [ ]:
# HISTOGRAMAS de las variables numéricas
df[num_cols].hist(figsize=(15, 10), bins=30)
plt.tight_layout()
plt.show()


In [ ]:
# BOXPLOTS de LIMIT_BAL para detectar posibles outliers
sns.boxplot(x=df['LIMIT_BAL'])
plt.show()

In [ ]:
# Cálculo del IQR para LIMIT_BAL
Q1 = df['LIMIT_BAL'].quantile(0.25)
Q3 = df['LIMIT_BAL'].quantile(0.75)
IQR = Q3 - Q1

filtro = (df['LIMIT_BAL'] >= Q1 - 1.5 * IQR) & (
    df['LIMIT_BAL'] <= Q3 + 1.5 * IQR)

# Mostrar valores atípicos
outliers = df[~filtro]
print(outliers)

In [ ]:
# CORRELACIÓN entre variables numéricas
plt.figure(figsize=(12, 10))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Matriz de Correlación')
plt.show()

In [ ]:
# PAIRPLOT de variables relevantes
rel_cols = ['LIMIT_BAL', 'AGE', 
            'BILL_AMT1', 'BILL_AMT2',
            'PAY_AMT1', 'PAY_AMT2'] 

sns.pairplot(df[rel_cols])
plt.show()

## Feature Engineering

**Es crear nuevas variables que mejoren la capacidad del modelo para aprender patrones**

* No es limpiar datos.
* No es escalar.
* Es crear inteligencia a partir de los datos.

In [ ]:
# Ratio deuda / límite (clave en riesgo)
df['deb_ratio']= df['BILL_AMT1']/df['LIMIT_BAL']

# Promedio de deuda
df['avg_bill']= df[[
    'BILL_AMT1','BILL_AMT2','BILL_AMT3',
    'BILL_AMT4','BILL_AMT5','BILL_AMT6'
]].mean(axis=1)

# Promedio de pago
df['avg_pay']= df[[
    'PAY_AMT1','PAY_AMT2','PAY_AMT3',
    'PAY_AMT4','PAY_AMT5','PAY_AMT6'
]].mean(axis=1)

# Ratio pago/deuda
df['pay_to_bill_ratio']= df['avg_pay']/(df['avg_bill']+1)

# Número de retrasos (muy importante)
df['num_delays']= (df[[
    'PAY_0','PAY_2','PAY_3',
    'PAY_4','PAY_5','PAY_6',
]] >0 ).sum(axis=1)

df.head()


> 👉 Antes de eliminar outliers, pregúntate:

¿Son errores?\
¿Son casos raros pero válidos?\
¿Me conviene tratarlos o escalarlos después?\
En este caso, probablemente NO debes eliminarlos. Solo analizarlos.

In [ ]:
# Rellenar valores NaN con 0
df['pay_to_bill_ratio'] = df['pay_to_bill_ratio'].fillna(0)

## Split

Ahora mismo hiciste esto:

1. Escalaste todo el dataframe
2. Luego hiciste train/test split

> Eso en proyectos serios se considera data leakage, porque el scaler vio datos del test.

In [ ]:
# Eliminamos la variable objetivo y dividimos en train y test.
X = df.drop('default_payment_next_month', axis=1)
y = df['default_payment_next_month']

# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Distribución train:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"Distribución test:\n{y_test.value_counts(normalize=True).round(3)}")

## Escalado

In [ ]:
# ---- StandardScaler ----
# Copiamos los datos para no modificar los originales
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler_S = StandardScaler()

# Escalamos solo las columnas numéricas
X_train_scaled[num_cols] = scaler_S.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler_S.transform(X_test[num_cols])

print("Train:", X_train_scaled.shape)
print("Test:", X_test_scaled.shape)


In [ ]:
# ---- RobustScaler ----
X_train_R = X_train.copy()
X_test_R = X_test.copy()

scaler_R = RobustScaler()

X_train_R[num_cols] = scaler_R.fit_transform(X_train_R[num_cols])
X_test_R[num_cols] = scaler_R.transform(X_test_R[num_cols])

print("Test:", X_test_R.shape)


In [ ]:
# Añadir las variables categóricas al conjunto escalado (si es necesario)
# En este caso, las dejamos sin escalar porque son binarias o de conteo.

categoricas = ['SEX', 'EDUCATION', 'MARRIAGE',
               'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

X_train_scaled[categoricas] = X_train[categoricas]
X_test_scaled[categoricas] = X_test[categoricas]

X_train_R[categoricas] = X_train[categoricas]
X_test_R[categoricas] = X_test[categoricas]

> Nunca uses el dataset completo después de dividir

## 8. Reducción de Dimensionalidad (Opcional)

Primero escalas, luego reduces dimensión.
Nunca al revés.

PCA DEBE hacerse con datos escalados, Porque PCA es sensible a escala. (X_train_scaled)

In [ ]:
# Reducción de dimensionalidad con PCA
# Aplicar PCA para reducir a 2 dimensiones

pca = PCA(n_components=2)

X_train_pca = pca.fit_transform(X_train_R[num_cols])
X_test_pca = pca.transform(X_test_R[num_cols])

print("Varianza explicada:", pca.explained_variance_ratio_.round(2))
print("Varianza acumulada:", pca.explained_variance_ratio_.cumsum().round(2))


el primer componente explica el 61% de la variabilidad y el segundo el 30%, juntos capturan el 91% de la información original.


In [ ]:
# Gráfica de varianza explicada acumulada para PCA
pca = PCA()
pca.fit(X_train_R[num_cols])
plt.plot(pca.explained_variance_ratio_.cumsum(), marker='o')
plt.title('Varianza Explicada Acumulada por PCA')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Explicada Acumulada')
plt.grid(True)
plt.show()


**n_neighbors**: *controla el tamaño del vecindario local.*\
bajo → clusters más separados\
alto → estructura global

**min_dist**: *qué tan compactos son los clusters.*\
bajo → clusters densos\
alto → clusters más dispersos

Tu configuración es bastante estándar.

In [ ]:
# Aplicar UMAP para reducir a 2 dimensiones
reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_train_R[num_cols])

print("Datos reducidos con UMAP:")
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_train, cmap='viridis', s=50)
plt.colorbar(label='Default Payment Next Month')
plt.title('UMAP de Clientes')
plt.grid(True)
plt.show()

In [ ]:
''''Reducción de dimensionalidad: X_pca y X_umap'''
print(f"Formato original: {df.shape}")
print(f"Formato reduzidocon PCA: {X_train_pca.shape}")
print(f"Formato reduzidocon umap: {X_umap.shape}")

## 9. MODELOS

### APRENDIZAJE SUPERVISADO

**❓ ¿Por qué se inicializa todo junto?**\
👉 Porque le dices: “Aplica esto en orden automáticamente”

In [ ]:
'''
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('model', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
'''

#### 9.1 Clasificación

In [ ]:
#---------- Modelo sin escalado ----------
model_simple= LogisticRegression(max_iter= 1000)
model_simple.fit(X_train, y_train)
y_pred_simple = model_simple.predict(X_test)

# --------- Modelo StandardScaler --------
model_scaled= LogisticRegression(max_iter= 1000)
model_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = model_scaled.predict(X_test_scaled)

#---------- Modelo RobustScaler ----------
model_robust= LogisticRegression(max_iter= 1000)
model_robust.fit(X_train_R, y_train)
y_pred_robust = model_robust.predict(X_test_R)

#----------- Modelo con PCA -------------
model_pca= LogisticRegression(max_iter= 1000)
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)


#### 9.2 Regresión

In [ ]:
# 👉 Variable objetivo de regresión
y_reg= df['avg_bill']

> Quitar BILL_AMT del modelo de regresión\
> Crear variable objetivo más independiente

Evaluar con:
* Recall
* Precision
* ROC AUC

In [ ]:
#---------- Modelo sin escalado ----------
reg_simple= LinearRegression()
reg_simple.fit(X_train, y_reg.loc[X_train.index])
y_pred_reg_simple = reg_simple.predict(X_test)

# --------- Modelo StandardScaler --------
reg_scaled= LinearRegression()
reg_scaled.fit(X_train_scaled, y_reg.loc[X_train.index])
y_pred_reg_scaled = reg_scaled.predict(X_test_scaled)

#---------- Modelo RobustScaler ----------
reg_robust= LinearRegression()
reg_robust.fit(X_train_R, y_reg.loc[X_train.index])
y_pred_reg_robust = reg_robust.predict(X_test_R)

#----------- Modelo con PCA -------------
reg_pca= LinearRegression()
reg_pca.fit(X_train_pca, y_reg.loc[X_train.index])
y_pred_reg_pca = reg_pca.predict(X_test_pca)

### APRENDIZAJE NO SUPERVISADO

#### 9.3. Clustering

##### n_clusters

In [ ]:
# Evaluar el mejor número de clusters con silhouette score
sil_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_umap)
    score = silhouette_score(X_umap, labels)
    sil_scores.append(score)
    print(f"K={k}, Silhouette Score: {score:.2f}")

plt.plot(range(2, 11), sil_scores, marker='o')
plt.title('Silhouette Score para K-Means con UMAP')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Silhouette Score')
plt.show()

In [ ]:
# Método Elbow Method para determinar el número óptimo de clusters en K-Means
inertia = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_train_pca)
    inertia.append(kmeans.inertia_)

plt.plot(range(1, 11), inertia, marker='o')
plt.title('Método del Codo para K-Means')
plt.xlabel('Número de Clusters')
plt.ylabel('Inercia')
plt.grid(True)
plt.show()

> ¿Cuánta información de los datos originales conserva el PCA?

##### Modelos

In [ ]:

# Definir el número de clusters para K-Means
n_clusters = 4  # Basado en el análisis de silhouette score y elbow method
kmeans = KMeans(n_clusters=n_clusters, random_state=42)

# Ajustar el modelo K-Means a los datos reducidos con UMAP
kmeans.fit(X_umap)

# Obtener las etiquetas de los clusters
labels_kmeans = kmeans.labels_

# Visualizar los clusters obtenidos con K-Means
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=labels_kmeans, cmap='viridis', s=50)
plt.title('Clusters obtenidos con K-Means')
plt.grid(True)
plt.show() 


In [ ]:
# Definir los parámetros para DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=4)

# Ajustar el modelo DBSCAN a los datos reducidos con UMAP
dbscan.fit(X_umap)

# Obtener las etiquetas de los clusters
labels_dbscan = dbscan.labels_

# Visualizar los clusters obtenidos con DBSCAN
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=labels_dbscan, cmap='viridis', s=50)
plt.title('Clusters obtenidos con DBSCAN')
plt.grid(True)
plt.show()


In [ ]:
# Definir el numero de clusters para Agglomerative Clustering
hierarchical = AgglomerativeClustering(n_clusters=n_clusters)

# Ajustar el modelo de clustering jerárquico a los datos reducidos con UMAP
labels_hierarchical = hierarchical.fit_predict(X_umap)

# Visualizar los clusters obtenidos con Agglomerative Clustering
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=labels_hierarchical, cmap='viridis', s=50)
plt.title('Clusters obtenidos con Agglomerative Clustering')
plt.grid(True)
plt.show()


## 10. Evaluación

### APRENDIZAJE SUPERVISADO

#### 10.1 Clasificación

In [ ]:
print("Simple:", accuracy_score(y_test, y_pred_simple))
print("Standard:", accuracy_score(y_test, y_pred_scaled))
print("Robust:", accuracy_score(y_test, y_pred_robust))
print("PCA:", accuracy_score(y_test, y_pred_pca))

In [ ]:
'''
from sklearn.metrics import roc_curve, auc
y_prob =model_robust.predict_proba(X_test_R)[:,1]

fpr, tpr = roc_curve(y_test, y_prob)
rock_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)
plt.title(f'ROC Curve (AUC = {rock_auc:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.grid()
plt.show()
'''

#### 10.2 Regresión

In [ ]:
import numpy as np

print("RMSE Simple:", np.sqrt(mean_squared_error(y_reg.loc[X_test.index], y_pred_reg_simple)))
print("RMSE Standard:", np.sqrt(mean_squared_error(y_reg.loc[X_test.index], y_pred_reg_scaled)))
print("RMSE Robust:", np.sqrt(mean_squared_error(y_reg.loc[X_test.index], y_pred_reg_robust)))
print("RMSE PCA:", np.sqrt(mean_squared_error(y_reg.loc[X_test.index], y_pred_reg_pca)))

### APRENDIZAJE NO SUPERVISADO

#### 10.3 clustering

In [ ]:
# Silhouette
silhouette_kmeans = silhouette_score(X_umap, labels_kmeans)
print(f"Silhouette Score para K-Means: {silhouette_kmeans:.2f}")

# quitar ruido de DBSCAN (etiqueta -1)
mask_dbscan = labels_dbscan != -1

silhouette_dbscan = silhouette_score(
    X_umap[mask_dbscan], 
    labels_dbscan[mask_dbscan]
    )
print(f"Silhouette Score para DBSCAN: {silhouette_dbscan:.2f}")

silhouette_hierarchical = silhouette_score(X_umap, labels_hierarchical)
print(f"Silhouette Score para Agglomerative Clustering: {silhouette_hierarchical:.2f}")


In [ ]:
# Davies-Bouldin Index
daviesB_kmeans = davies_bouldin_score(X_umap, labels_kmeans)
print(f"Davies-Bouldin Index para K-Means: {daviesB_kmeans:.2f}")

daviesB_dbscan = davies_bouldin_score(X_umap, labels_dbscan)
print(f"Davies-Bouldin Index para DBSCAN: {daviesB_dbscan:.2f}")

daviesB_hierarchical = davies_bouldin_score(X_umap, labels_hierarchical)
print(f"Davies-Bouldin Index para Agglomerative Clustering: {daviesB_hierarchical:.2f}")


In [ ]:
# Calinski-Harabasz 
calinski_kmeans = calinski_harabasz_score(X_umap, labels_kmeans)
print(f"Calinski-Harabasz Index para K- Means: {calinski_kmeans:.2f}")

calinski_dbscan = calinski_harabasz_score(X_umap, labels_dbscan)
print(f"Calinski-Harabasz Index para DBSCAN: {calinski_dbscan:.2f}")

calinski_hierarchical = calinski_harabasz_score(X_umap, labels_hierarchical)
print(f"Calinski-Harabasz Index para Agglomerative Clustering: {calinski_hierarchical:.2f}")


In [ ]:
# Evaluar si los clusters separan defaulters de no defaulters
# Si un cluster tiene muchos más defaults
# significa que el clustering está capturando riesgo crediticio.

KM = pd.crosstab(labels_kmeans, y_train)
print("Crosstab para K-Means:", KM)

DB = pd.crosstab(labels_dbscan, y_train)
print("Crosstab para DBSCAN:", DB)

Hierarchical = pd.crosstab(labels_hierarchical, y_train)
print("Crosstab para Agglomerative Clustering:", Hierarchical)


> El k con mayor silhouette suele ser el mejor.

## 11. Interpretación de clusters

Este es el paso que diferencia a un estudiante de un analista real.

Debes responder:

¿Qué tipo de clientes forman cada cluster?

¿Cuál parece más riesgoso?

¿Cuál más estable?

¿Cómo usaría el banco esta segmentación?

Si no interpretas, el proyecto queda incompleto.

In [ ]:
# Interpretar Clusters
# Analizar las características promedio de cada cluster para entender su perfil.

# Cluster 0 -> Clientes con altos límites de crédito
# Cluster 1 -> Clientes con retrasos frecuentes
# Cluster 2 -> Clientes de bajo riesgo



📊 3. Visualizaciones (ENFOQUE NEGOCIO)

No hagas gráficas bonitas. Haz gráficas que respondan:

Preguntas clave:
* ¿Quiénes incumplen más?
* ¿Qué patrón tienen?
* ¿El retraso predice default?
* ¿Relación deuda vs pago?

📦 4. GitHub (NO OPCIONAL)

Estructura:

uci_credit_risk\
├── data\
├── notebooks\
├── src\
├── README.md

README debe tener:

* Problema de negocio
* Dataset
* Insights clave (NO código)
* Conclusión tipo empresa